# Capitolul 7: Crearea aplicațiilor de chat
## Start rapid cu API-ul OpenAI

Acest caiet este adaptat din [Depozitul de exemple Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) care include caiete ce accesează serviciile [Azure OpenAI](notebook-azure-openai.ipynb).

API-ul Python OpenAI funcționează și cu modelele Azure OpenAI, cu câteva modificări. Aflați mai multe despre diferențe aici: [Cum să comutați între endpoint-urile OpenAI și Azure OpenAI cu Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Prezentare generală  
"Modelele mari de limbaj sunt funcții care transformă text în text. Având un șir de text de intrare, un model mare de limbaj încearcă să prezică textul care va urma"(1). Acest caiet "quickstart" va introduce utilizatorii în conceptele de nivel înalt ale LLM, cerințele esențiale ale pachetului pentru a începe cu AML, o introducere ușoară în proiectarea prompturilor și câteva exemple scurte de diferite cazuri de utilizare. 


## Table of Contents  

[Overview](#overview)  
[How to use OpenAI Service](#how-to-use-openai-service)  
[1. Creating your OpenAI Service](#1.-creating-your-openai-service)  
[2. Installation](#2.-installation)    
[3. Credentials](#3.-credentials)  

[Use Cases](#use-cases)    
[1. Summarize Text](#1.-summarize-text)  
[2. Classify Text](#2.-classify-text)  
[3. Generate New Product Names](#3.-generate-new-product-names)  
[4. Fine Tune a Classifier](#4.fine-tune-a-classifier)  

[References](#references)


### Construiește primul tău prompt  
Acest exercițiu scurt va oferi o introducere de bază pentru trimiterea prompturilor către un model OpenAI pentru o sarcină simplă de "rezumat".  


**Pași**:  
1. Instalează biblioteca OpenAI în mediul tău Python  
2. Încarcă bibliotecile standard de ajutor și configurează-ți acreditările obișnuite de securitate OpenAI pentru serviciul OpenAI pe care l-ai creat  
3. Alege un model pentru sarcina ta  
4. Creează un prompt simplu pentru model  
5. Trimite cererea ta către API-ul modelului!  


### 1. Instalarea OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importă bibliotecile ajutătoare și instanțiază acreditările


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Găsirea modelului potrivit  
Modele precum GPT-4o și GPT-4o mini pot înțelege și genera limbaj natural și sunt disponibile pe platforma OpenAI cu diferite niveluri de putere și viteză, potrivite pentru diferite sarcini.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Proiectarea promptului  

"Magia modelelor lingvistice mari constă în faptul că, fiind antrenate să minimizeze această eroare de predicție pe cantități vaste de text, modelele ajung să învețe concepte utile pentru aceste predicții. De exemplu, ele învață concepte precum"(1):

* cum să se scrie corect
* cum funcționează gramatica
* cum să parafrazeze
* cum să răspundă la întrebări
* cum să poarte o conversație
* cum să scrie în multe limbi
* cum să programeze
* etc.

#### Cum să controlezi un model lingvistic mare  
"Dintre toate intrările către un model lingvistic mare, de departe cea mai influentă este promptul text(1).

Modelele lingvistice mari pot fi ghidate să producă ieșiri în câteva moduri:

Instrucțiune: Spune modelului ce dorești
Completare: Induce modelul să completeze începutul a ceea ce vrei
Demonstrație: Arată modelului ce dorești, cu:
Câteva exemple în prompt
Sute sau mii de exemple într-un set de date de antrenament pentru ajustare fină"



#### Există trei reguli de bază pentru crearea prompturilor:

**Arată și spune**. Fă clar ce dorești fie prin instrucțiuni, exemple sau o combinație a celor două. Dacă vrei ca modelul să clasifice o listă de elemente în ordine alfabetică sau să clasifice un paragraf după sentiment, arată-i că asta vrei.

**Oferă date de calitate**. Dacă încerci să construiești un clasificator sau să faci modelul să urmeze un tipar, asigură-te că există suficiente exemple. Verifică-ți exemplele — modelul este de obicei suficient de inteligent să vadă greșelile de ortografie de bază și să-ți ofere un răspuns, dar s-ar putea să considere că acestea sunt intenționate și să afecteze răspunsul.

**Verifică setările tale.** Setările temperature și top_p controlează cât de determinist este modelul în generarea unui răspuns. Dacă îi ceri un răspuns pentru care există un singur răspuns corect, atunci vei dori să nici le la valori scăzute. Dacă dorești răspunsuri mai diverse, atunci s-ar putea să le setezi la valori mai mari. Greșeala numărul unu pe care o fac oamenii cu aceste setări este să presupună că sunt controale pentru „inteligență” sau „creativitate”.


Sursa: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Trimite!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Repetă aceeași apelare, cum se compară rezultatele?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Rezumare text  
#### Provocare  
Rezumarea unui text prin adăugarea unui 'tl;dr:' la sfârșitul unui pasaj de text. Observați cum modelul înțelege să execute o serie de sarcini fără instrucțiuni suplimentare. Puteți experimenta cu prompturi mai descriptive decât tl;dr pentru a modifica comportamentul modelului și a personaliza rezumarea pe care o primiți(3).  

Munca recentă a demonstrat câștiguri substanțiale în multe sarcini și repere NLP prin pre-antrenarea pe un corpus mare de text urmată de reglaj fin pe o sarcină specifică. Deși, în general, arhitectura este independenta de sarcină, această metodă necesită totuși seturi de date de reglaj fin specifice sarcinii, cu mii sau zeci de mii de exemple. În contrast, oamenii pot de obicei să performeze o sarcină nouă de limbaj din doar câteva exemple sau din instrucțiuni simple – ceva ce sistemele NLP actuale încă întâmpină dificultăți majore să facă. Aici arătăm că scalarea modelelor de limbaj îmbunătățește mult performanța fără a ține cont de sarcină, în scenarii cu puține exemple, uneori atingând chiar competitivitatea cu abordările anterioare de reglaj fin de ultima generație.  



Tl;dr


# Exerciții pentru mai multe cazuri de utilizare  
1. Rezumă textul  
2. Clasifică textul  
3. Generează nume noi de produse


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Clasifică text  
#### Provocare  
Clasifică elementele în categoriile furnizate în timpul inferenței. În exemplul următor, oferim atât categoriile, cât și textul de clasificat în prompt (*playground_reference). 

Solicitare client: Bună, una dintre tastele de la tastatura laptopului meu s-a stricat recent și voi avea nevoie de o înlocuire:

Categorie clasificată:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Generează Nume Noi pentru Produse
#### Provocare
Creează nume de produse pornind de la cuvinte exemplu. Aici includem în prompt informații despre produsul pentru care urmează să generăm nume. De asemenea, oferim un exemplu similar pentru a arăta modelul pe care dorim să-l primim. Am setat, de asemenea, valoarea temperaturii ridicată pentru a crește aleatorietatea și a obține răspunsuri mai inovatoare.

Descriere produs: Un aparat de făcut milkshake-uri pentru acasă
Cuvinte de bază: rapid, sănătos, compact.
Nume de produse: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Descriere produs: O pereche de pantofi care se pot adapta oricărei mărimi de picior.
Cuvinte de bază: adaptabil, potrivit, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referințe  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Portalul Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Cele mai bune practici pentru ajustarea fină a modelelor GPT în clasificarea textului](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Pentru Mai Mult Ajutor  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Contributori
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
